# Phase 6 Worksheet — RAG Patterns
**Corrected in this version:** every `multimodal_chat()` call replaced with `ask()`, which actually routes to the requested model (e.g. `MODEL_LLAMA` as judge in other phases genuinely hits Llama's endpoint now, not Qwen3-14B's).

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))  # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=200):
    """Drop-in replacement for the old multimodal_chat() text-only calls --
    correctly routed per-model via get_chat_model(), unlike inhouse_llm.py's
    own chat()/multimodal_chat() which always hit the Qwen3-14B endpoint."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=200):
    """Drop-in replacement for multimodal_chat() WITH an image -- uses the
    corrected image_url content-block format, and an actual client for the
    vision model (inhouse_llm.py never created one)."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

import chromadb
client = chromadb.HttpClient(host="localhost", port=8000)  # adjust to your Chroma server
print("Setup OK")

In [ ]:
collection = client.get_or_create_collection("phase6_patterns")
docs = {
    "auth_a": "API A uses OAuth2 bearer tokens for authentication, with a rate limit of 100 requests per minute.",
    "auth_b": "API B uses static API keys for authentication, with a rate limit of 50 requests per minute.",
}
ids = list(docs.keys())
texts = list(docs.values())
collection.upsert(ids=ids, embeddings=embedder.embed_documents(texts), documents=texts)

## 1. Naive RAG

In [ ]:
def naive_rag(question, k=2):
    q_vec = embedder.embed_query(question)
    chunks = collection.query(query_embeddings=[q_vec], n_results=k)["documents"][0]
    context = "\n".join(chunks)
    return ask("Answer using only the context.", f"Context: {context}\nQuestion: {question}", max_tokens=200)

print(naive_rag("What auth method does API A use?"))

## 2. Naive Agentic RAG vs decomposed (this phase's teaser, solved)

In [ ]:
compound_question = "Compare the auth method and rate limits of APIs A and B"

print("--- Naive (single retrieval call) ---")
print(naive_rag(compound_question, k=2))

print("\n--- Decomposed (multi-hop style) ---")
sub_queries = ["auth method of API A", "rate limit of API A", "auth method of API B", "rate limit of API B"]
all_chunks = []
for sq in sub_queries:
    q_vec = embedder.embed_query(sq)
    all_chunks.extend(collection.query(query_embeddings=[q_vec], n_results=1)["documents"][0])
context = "\n".join(set(all_chunks))
print(ask("Answer using only the context.", f"Context: {context}\nQuestion: {compound_question}", max_tokens=250))

## 3. Corrective RAG (CRAG)

In [ ]:
def crag(question, k=2):
    q_vec = embedder.embed_query(question)
    chunks = collection.query(query_embeddings=[q_vec], n_results=k)["documents"][0]
    context = "\n".join(chunks)

    judgment = ask(
        "Are these chunks relevant to the question? Reply ONLY 'relevant' or 'not_relevant'.",
        f"Question: {question}\nChunks: {context}", max_tokens=20,
    )
    if "not_relevant" in judgment.lower():
        return f"[CORRECTIVE ACTION TRIGGERED] Retrieval looked weak for: '{question}'. Falling back to a broader search or flagging for the user."
    return ask("Answer using only the context.", f"Context: {context}\nQuestion: {question}", max_tokens=200)

print(crag("What auth method does API A use?"))
print("\n", crag("What is the weather like today?"))  # should trigger corrective action

## 4. Graph RAG alongside Chroma

In [ ]:
import networkx as nx

g = nx.DiGraph()
g.add_edge("API A", "OAuth2", relation="uses_auth")
g.add_edge("API B", "API_Key", relation="uses_auth")
g.add_edge("API A", "Payment Service", relation="depends_on")

def graph_rag(entity):
    chroma_result = collection.query(query_embeddings=[embedder.embed_query(entity)], n_results=1)
    graph_facts = list(g.edges(entity, data=True)) if entity in g else []
    return chroma_result["documents"][0], graph_facts

text_match, relations = graph_rag("API A")
print("Chroma text match:", text_match)
print("Graph relations:", relations)

## Teaser exercise
Build Adaptive RAG: write a router that classifies a question as 'factual' (use naive_rag) vs 'comparison' (use the decomposed multi-hop version from section 2) vs 'relationship' (use graph_rag), then dispatches automatically. Use `ask(..., model=MODEL_QWEN3_30B)` for the router itself, since classification benefits from the stronger reasoning model.